<a href="https://colab.research.google.com/github/MaggieHDez/ClassFiles/blob/IDA/transformaciones_pySpark_255879.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Práctica: Transformaciones con PySpark - NSL_KDD.csv
>Clase: Ingeniería de Datos Avanzada\
>Alumno: Margarita Cristina Hernández Delgadillo\
>Matrícula: 255879\
>Fecha: 15 de Mayo de 2026

In [59]:
# =========================================================
# Montar Google Drive
# =========================================================

from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Instalación de librerías necesarias

In [60]:

!pip install pyspark -q

## 2. Verificación de versión de PySpark

In [61]:
# Verificar la versión e imprimir
print("\nVersión de PySpark:")
!pyspark --version


Versión de PySpark:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.2
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.18
Branch HEAD
Compiled by user runner on 2026-02-02T08:08:13Z
Revision 7cc3b9bcdaab8c923f23cdbc9ce922530e1becf1
Url https://github.com/apache/spark
Type --help for more information.


## 3. Creación de sesión de Spark

In [62]:
# =========================================================
# 3. Creación de la sesión de Spark
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Practica_NSL_KDD")
    .getOrCreate()
)

sc = spark.sparkContext

if sc:
    print(f"Spark Version: {sc.version}")
    print("SparkContext creado exitosamente")
else:
    print("No se pudo crear el SparkContext")

Spark Version: 4.0.2
SparkContext creado exitosamente


## 4. Carga y exploración inicial del dataset NSL_KDD.csv

En esta primera parte se cargó el archivo `NSL_KDD.csv` desde Google Drive
en un DataFrame de PySpark. También se revisó el esquema, los tipos de datos,
el número total de filas, el número total de columnas y las primeras 10 filas.

Esto permite verificar que el archivo se cargó correctamente y que las columnas
necesarias están disponibles para realizar las transformaciones solicitadas.

### 4.1 Carga del dataset NSL_KDD.csv

In [63]:
# Ruta del archivo CSV en Google Drive
ruta_archivo = "/content/drive/MyDrive/ClassFilesIDA/NSL_KDD.csv"

# Cargar el archivo CSV en un DataFrame de PySpark
# header = true usa la primera fila como nombres de columnas
# inferSchema = true detecta automáticamente los tipos de datos
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(ruta_archivo)
)

print("Dataset cargado correctamente")

Dataset cargado correctamente


### 4.2 Exploración inicial del DataFrame original

#### 4.2.1 Código

In [64]:
# Mostrar el esquema del DataFrame
print("\nSchema:")
df.printSchema()

# Mostrar los tipos de datos de cada columna
print("\nTipos de datos:")
print(df.dtypes)

# Número total de filas
print("\nNúmero total de filas:")
print(df.count())

# Número total de columnas
print("\nNúmero total de columnas:")
print(len(df.columns))

# Mostrar las primeras 10 filas
print("\nPrimeras 10 filas:")
df.show(10)


Schema:
root
 |-- duration: string (nullable = true)
 |-- protocol_type: string (nullable = true)
 |-- service: string (nullable = true)
 |-- flag: integer (nullable = true)
 |-- src_bytes: integer (nullable = true)
 |-- dst_bytes: integer (nullable = true)
 |-- land: integer (nullable = true)
 |-- wrong_fragment: integer (nullable = true)
 |-- urgent: integer (nullable = true)
 |-- hot: integer (nullable = true)
 |-- num_failed_logins: integer (nullable = true)
 |-- logged_in: integer (nullable = true)
 |-- num_compromised: integer (nullable = true)
 |-- root_shell: integer (nullable = true)
 |-- su_attempted: integer (nullable = true)
 |-- num_root: integer (nullable = true)
 |-- num_file_creations: integer (nullable = true)
 |-- num_shells: integer (nullable = true)
 |-- num_access_files: integer (nullable = true)
 |-- num_outbound_cmds: integer (nullable = true)
 |-- is_host_login: integer (nullable = true)
 |-- is_guest_login: integer (nullable = true)
 |-- count: integer (nullab

#### 4.2.2 Análisis de la exploración inicial

En la exploración inicial se observó que el archivo se cargó correctamente,
pero los valores no coinciden con los nombres de las columnas.

Por ejemplo, el valor `tcp` aparece debajo de la columna `duration`, aunque
realmente corresponde a `protocol_type`. También se observa que el archivo
contiene una columna llamada `class`, pero en las instrucciones de la práctica
esta columna se solicita con el nombre `label`.

Esto indica dos ajustes necesarios antes de continuar:
1. Corregir los nombres de las columnas por posición, ya que los datos están
   recorridos con respecto al encabezado.
2. Renombrar la columna `class` como `label` para que coincida con las instrucciones
   de la práctica.

Después de estos ajustes, el DataFrame quedará listo para aplicar las
transformaciones solicitadas.

## 5. Corrección de columnas movidas y renombrado de class a label

### 5.1 Código

In [65]:

# Guardar los nombres originales de las columnas
columnas_originales = df.columns
primer_columna = columnas_originales[0]

# Mostrar el nombre de la primera columna
print("Nombre de la primera columna:", primer_columna)

# Como los datos están recorridos, se elimina el primer nombre de columna
# y se agrega duration al final para conservar el último valor del registro
columnas_corregidas = columnas_originales[1:] + [primer_columna]

# Aplicar los nombres corregidos al DataFrame
df = df.toDF(*columnas_corregidas)

# La práctica solicita una columna llamada label.
# En el archivo esta columna aparece como class, por eso se renombra.
df = df.withColumnRenamed("class", "label")

print("Columnas corregidas:")
for i, columna in enumerate(df.columns):
    print(i, columna)

print("\nPrimeras 10 filas después de corregir columnas:")
df.show(10)

Nombre de la primera columna: duration
Columnas corregidas:
0 protocol_type
1 service
2 flag
3 src_bytes
4 dst_bytes
5 land
6 wrong_fragment
7 urgent
8 hot
9 num_failed_logins
10 logged_in
11 num_compromised
12 root_shell
13 su_attempted
14 num_root
15 num_file_creations
16 num_shells
17 num_access_files
18 num_outbound_cmds
19 is_host_login
20 is_guest_login
21 count
22 srv_count
23 serror_rate
24 srv_serror_rate
25 rerror_rate
26 srv_rerror_rate
27 same_srv_rate
28 diff_srv_rate
29 srv_diff_host_rate
30 dst_host_count
31 dst_host_srv_count
32 dst_host_same_srv_rate
33 dst_host_diff_srv_rate
34 dst_host_same_src_port_rate
35 dst_host_srv_diff_host_rate
36 dst_host_serror_rate
37 dst_host_srv_serror_rate
38 dst_host_rerror_rate
39 dst_host_srv_rerror_rate
40 label
41 duration

Primeras 10 filas después de corregir columnas:
+-------------+-------+----+---------+---------+----+--------------+------+---+-----------------+---------+---------------+----------+------------+--------+--------

### 5.2 Análisis de la corrección de columnas

Después de corregir los nombres de las columnas, los valores quedaron
alineados correctamente. Ahora `tcp` aparece en `protocol_type`, `private` aparece
en `service` y `REJ` aparece en `flag`.

También se renombró la columna `class` como `label`, ya que las instrucciones
de la práctica solicitan trabajar con una columna llamada `label`.

A partir de este DataFrame corregido ya se pueden realizar las transformaciones
solicitadas.

## 6. Selección de columnas específicas

### 6.1 Código

In [66]:
# Seleccionar únicamente las columnas solicitadas en la práctica
df_seleccionado = df.select(
    "protocol_type",
    "service",
    "src_bytes",
    "dst_bytes",
    "label"
)

# Mostrar las primeras 10 filas del DataFrame seleccionado
print("Columnas seleccionadas:")
df_seleccionado.show(10)

Columnas seleccionadas:
+-------------+-------+---------+---------+------------+
|protocol_type|service|src_bytes|dst_bytes|       label|
+-------------+-------+---------+---------+------------+
|          tcp|private|        0|        0|     neptune|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|       15|       mscan|
|          tcp|   http|      267|    14515|      normal|
|          tcp| telnet|      129|      174|guess_passwd|
|          tcp|   http|      327|      467|      normal|
|          tcp|    ftp|       26|      157|guess_passwd|
|          tcp| telnet|        0|        0|       mscan|
|          tcp|private|        0|        0|     neptune|
|          tcp| telnet|        0|        0|     neptune|
+-------------+-------+---------+---------+------------+
only showing top 10 rows


### 6.2 Análisis de la selección de columnas

En esta transformación se seleccionaron únicamente las columnas solicitadas
en la práctica: `protocol_type`, `service`, `src_bytes`, `dst_bytes` y `label`.

Después de la corrección realizada previamente, estas columnas ya se encuentran
alineadas correctamente. Por ejemplo, `protocol_type` contiene el tipo de protocolo,
`service` contiene el servicio utilizado, `src_bytes` y `dst_bytes` contienen los bytes
de origen y destino, y `label` contiene la clase o etiqueta del registro.

Esta selección permite trabajar con un DataFrame más pequeño y enfocado solo en
las variables necesarias para las siguientes transformaciones

## 7. Filtrado de registros `protocol_type = 'tcp'` y `src_bytes > 1000`

### 7.1 Código

In [67]:
# Filtrar únicamente los registros donde protocol_type sea tcp
# y src_bytes sea mayor a 1000
df_filtrado = df_seleccionado.filter(
    (F.col("protocol_type") == "tcp") & (F.col("src_bytes") > 1000)
)

# Mostrar las primeras 10 filas del DataFrame filtrado
print("Registros filtrados:")
df_filtrado.show(10)

# Mostrar el número de registros que cumplen con la condición
print("Número de registros filtrados:")
print(df_filtrado.count())

Registros filtrados:
+-------------+--------+---------+---------+-----------+
|protocol_type| service|src_bytes|dst_bytes|      label|
+-------------+--------+---------+---------+-----------+
|          tcp|    http|    76944|        1|    apache2|
|          tcp|ftp_data|   283618|        0|warezmaster|
|          tcp|    http|    72564|        0|    apache2|
|          tcp|    smtp|     2599|      293|   mailbomb|
|          tcp|    smtp|     4030|      332|     normal|
|          tcp|    http|    72564|        0|    apache2|
|          tcp|    http|    54540|     8314|       back|
|          tcp|    smtp|     1376|      343|     normal|
|          tcp|    smtp|     1500|      332|     normal|
|          tcp|    smtp|     1710|      366|     normal|
+-------------+--------+---------+---------+-----------+
only showing top 10 rows
Número de registros filtrados:
1552


### 7.2 Análisis del filtrado

En esta transformación se filtraron únicamente los registros donde
`protocol_type` es igual a tcp y `src_bytes` es mayor a 1000.

El resultado muestra solo conexiones TCP con una cantidad de bytes de origen
mayor al límite indicado. Esto permite reducir el conjunto de datos y enfocarse
en registros con mayor tráfico enviado desde el origen.

## 8. Creación de la columna `total_bytes`

### 8.1 Código

In [68]:
# Crear una nueva columna llamada total_bytes
# Esta columna contiene la suma de src_bytes + dst_bytes
df_total_bytes = df_filtrado.withColumn(
    "total_bytes",
    F.col("src_bytes") + F.col("dst_bytes")
)

# Mostrar las primeras 10 filas con la nueva columna
print("DataFrame con la nueva columna total_bytes:")
df_total_bytes.show(10)

DataFrame con la nueva columna total_bytes:
+-------------+--------+---------+---------+-----------+-----------+
|protocol_type| service|src_bytes|dst_bytes|      label|total_bytes|
+-------------+--------+---------+---------+-----------+-----------+
|          tcp|    http|    76944|        1|    apache2|      76945|
|          tcp|ftp_data|   283618|        0|warezmaster|     283618|
|          tcp|    http|    72564|        0|    apache2|      72564|
|          tcp|    smtp|     2599|      293|   mailbomb|       2892|
|          tcp|    smtp|     4030|      332|     normal|       4362|
|          tcp|    http|    72564|        0|    apache2|      72564|
|          tcp|    http|    54540|     8314|       back|      62854|
|          tcp|    smtp|     1376|      343|     normal|       1719|
|          tcp|    smtp|     1500|      332|     normal|       1832|
|          tcp|    smtp|     1710|      366|     normal|       2076|
+-------------+--------+---------+---------+-----------+---

### 8.2 Análisis de la creación de total_bytes

En esta transformación se creó una nueva columna llamada `total_bytes`,
la cual contiene la suma de `src_bytes` y `dst_bytes`.

Esta columna permite conocer el total de bytes asociados a cada registro,
considerando tanto los bytes enviados desde el origen como los bytes recibidos
en el destino.

Con esta nueva variable es posible identificar conexiones con mayor volumen
total de tráfico.

## 9. Ordenamiento de registros por total_bytes

### 9.1 Código

In [69]:
# Ordenar los registros de mayor a menor usando la columna total_bytes
df_ordenado = df_total_bytes.orderBy(
    F.col("total_bytes").desc()
)

# Mostrar las primeras 10 filas ordenadas
print("Registros ordenados de mayor a menor por total_bytes:")
df_ordenado.show(10)

Registros ordenados de mayor a menor por total_bytes:
+-------------+--------+---------+---------+------+-----------+
|protocol_type| service|src_bytes|dst_bytes| label|total_bytes|
+-------------+--------+---------+---------+------+-----------+
|          tcp|     X11| 62825648|    90476| xlock|   62916124|
|          tcp|     X11| 31645608|   207796| xlock|   31853404|
|          tcp|ftp_data|  6291668|        0|normal|    6291668|
|          tcp|ftp_data|  3131464|        0|normal|    3131464|
|          tcp|ftp_data|  2194619|        0|normal|    2194619|
|          tcp|     X11|    13948|  1171108|normal|    1185056|
|          tcp|     X11|   286040|   383476|normal|     669516|
|          tcp|     X11|    39224|   511712| named|     550936|
|          tcp|ftp_data|   501760|        0|normal|     501760|
|          tcp|ftp_data|   501760|        0|normal|     501760|
+-------------+--------+---------+---------+------+-----------+
only showing top 10 rows


### 9.2 Análisis del ordenamiento

En esta transformación los registros fueron ordenados de mayor a menor
utilizando la columna `total_bytes`.

El resultado permite observar primero las conexiones con mayor cantidad total
de bytes. Esto puede ser útil para identificar registros con tráfico más alto
dentro del subconjunto filtrado previamente.

## 10. Valores únicos de `protocol_type`

### 10.1 Código

In [70]:
# Seleccionar los valores únicos de la columna protocol_type y mostrarlos
df.select(
    'protocol_type'
).distinct().show(truncate=False)

+-------------+
|protocol_type|
+-------------+
|tcp          |
|udp          |
|icmp         |
+-------------+



### 10.2 Análisis de la selección de valores únicos

En esta transformación se seleccionaron los valores únicos de la columna
`protocol_type`.

El resultado muestra que el dataset contiene tres tipos de protocolo: `tcp`,
`udp` e `icmp`. Esto permite conocer las categorías principales de conexión que
aparecen en los registros antes de realizar agrupaciones o conteos por protocolo.

## 11. Agrupación por `protocol_type`

### 11.1 Código

In [71]:
# Agrupar los registros por protocol_type
# y contar cuántos registros hay por cada tipo de protocolo
df.groupBy(
    "protocol_type"
).count().show(truncate=False)

+-------------+-----+
|protocol_type|count|
+-------------+-----+
|tcp          |15136|
|udp          |2074 |
|icmp         |825  |
+-------------+-----+



### 11.2 Análisis de agrupación

En esta transformación se agruparon los registros utilizando la columna `protocol_type` y se calculó el número de registros para cada tipo de protocolo.

El resultado muestra que el protocolo con mayor cantidad de registros es `tcp`, con 15136 registros. Después aparece `udp`, con 2074 registros, e `icmp`, con 825 registros.

Esto indica que la mayoría de las conexiones del dataset corresponden al protocolo `tcp`, mientras que `udp` e `icmp` tienen una presencia menor.

## 12. Agrupación por label y promedio de `src_bytes`

### 12.1 Código

In [72]:
# Agrupar los registros por label
# y calcular el promedio de src_bytes para cada etiqueta
df_promedio_label = df.groupBy(
    "label"
).agg(
    F.avg("src_bytes").alias("promedio_src_bytes")
)

# Mostrar los resultados ordenados de mayor a menor promedio
df_promedio_label.orderBy(
    F.col("promedio_src_bytes").desc()
).show(df_promedio_label.count(), truncate=False)

+---------------+-------------------+
|label          |promedio_src_bytes |
+---------------+-------------------+
|xlock          |1.350409942857143E7|
|warezmaster    |65186.258530183724 |
|rootkit        |54713.2            |
|back           |52894.35051546392  |
|apache2        |38689.75465313029  |
|named          |22001.64285714286  |
|worm           |4209.0             |
|xterm          |3137.909090909091  |
|mailbomb       |2599.0             |
|normal         |2501.925664398511  |
|buffer_overflow|1867.0             |
|multihop       |1767.8333333333333 |
|sendmail       |1698.7272727272727 |
|imap           |1352.0             |
|pod            |1341.2121212121212 |
|smurf          |909.5143403441682  |
|xsnoop         |775.75             |
|httptunnel     |506.83653846153845 |
|sqlattack      |398.0              |
|loadmodule     |277.0              |
|perl           |258.0              |
|ps             |124.58333333333333 |
|snmpgetattack  |106.52631578947368 |
|guess_passw

### 12.2 Análisis de agrupación por `label` y promedio de `src_bytes`

En esta transformación se agruparon los registros utilizando la columna `label` y se calculó el promedio de `src_bytes` para cada etiqueta.

El resultado fue ordenado de mayor a menor promedio. Se observa que la etiqueta `xlock` presenta el promedio más alto de `src_bytes`, seguida por `warezmaster`, `rootkit`, `back` y `apache2`.

También se observa que algunas etiquetas tienen un promedio de `src_bytes` igual a 0, como `neptune`, `nmap`, `portsweep`, `udpstorm` y `land`. Esto indica que, para esas etiquetas, los registros no presentan bytes enviados desde el origen o su valor promedio es cero.

La etiqueta `normal` también aparece en los resultados, con un promedio de `src_bytes` de aproximadamente 2501.93. Esto permite comparar el comportamiento de los registros normales frente a otras etiquetas del dataset.

## Cierre de Sesión

In [73]:
spark.stop()

print('✅ SparkSession finalizada correctamente.')

✅ SparkSession finalizada correctamente.
